# Phase 3 — Clinical Metadata + Fusion
15. Metadata MLP (fields confirmed: gender, gestational_age, birth_weight, plurality, delivery_mode)
16. Fusion (concat image_emb + metadata_emb -> linear/ReLU/dropout/linear) -> patient representation
17. Re-run few-shot (1/3/5-shot) with image+metadata
18. Results table: image-only vs image+metadata across shots

**Metadata fields used (confirmed from img_info.xlsx):**
- `gender` -> categorical (male/female)
- `gestational_age` -> numeric (weeks)
- `birth_weight` -> numeric (grams)
- `plurality` -> numeric/categorical (1 = singleton, 2 = twin, etc. -- treated as categorical since it's a small discrete set)
- `delivery_mode` -> categorical (vaginally / cesarean, etc.)

`categories` in img_info.xlsx duplicates the class label already used in the split CSVs -- not used as a feature.
`disease_description.xlsx` (disease/description text) is reserved for Phase 5 (text branch) -- not used here.

**Assumption:** `image` column in `img_info.xlsx` matches `image_path` in the split CSVs (same filename) -- used as the join key.

In [2]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler, OneHotEncoder

SEED = 42
np.random.seed(SEED)

EMB_PATH = r"E:\rop\codes\eyeclip_embeddings.zip"          # from Phase 1
IMG_INFO_PATH = r"E:\rop\dataset\ROP-VL\img_info.xlsx"              # CHANGE if stored elsewhere
SPLIT_DIR = Path(r"E:\rop\dataset\ROP-VL\split")  # CHANGE to actual path -- must match Phase 1's splits

META_NUMERIC = ["gestational_age", "birth_weight"]
META_CATEGORICAL = ["gender", "plurality", "delivery_mode"]

## Load embeddings + splits + metadata, align by image filename

**Explainability:** Phase 1's `.npz` stores embeddings in the row-order of `train.csv`/`val.csv`/`test.csv`. To attach metadata correctly we re-load those same CSVs (must be identical to Phase 1 -- same order, same rows) and join on `image_path` <-> `image`.

In [3]:
data = np.load(EMB_PATH)
X_train_img, y_train = data["X_train"], data["y_train"]
X_val_img,   y_val   = data["X_val"],   data["y_val"]
X_test_img,  y_test  = data["X_test"],  data["y_test"]

train_df = pd.read_csv(SPLIT_DIR / "train.csv")
val_df   = pd.read_csv(SPLIT_DIR / "val.csv")
test_df  = pd.read_csv(SPLIT_DIR / "test.csv")

# sanity: embeddings and split CSVs must have matching row counts, in order
assert len(train_df) == len(X_train_img), "train.csv row count != X_train row count -- splits changed since Phase 1?"
assert len(val_df)   == len(X_val_img),   "val.csv row count != X_val row count"
assert len(test_df)  == len(X_test_img),  "test.csv row count != X_test row count"

img_info = pd.read_excel(IMG_INFO_PATH)
print("img_info columns:", list(img_info.columns))
print("img_info missing values:\n", img_info[META_NUMERIC + META_CATEGORICAL].isna().sum())

img_info columns: ['image', 'patient_id', 'gender', 'gestational_age', 'birth_weight', 'plurality', 'delivery_mode', 'categories']
img_info missing values:
 gestational_age    0
birth_weight       0
gender             0
plurality          0
delivery_mode      0
dtype: int64


In [4]:
def attach_metadata(split_df, img_info):
    """Join split_df.image_path -> img_info.image, preserving split_df's row order.
    Input:  split_df [N rows] with 'image_path' column, img_info with 'image' + metadata columns
    Output: merged DataFrame, same length and order as split_df
    """
    merged = split_df.merge(img_info, left_on="image_path", right_on="image", how="left")
    n_missing = merged[META_NUMERIC + META_CATEGORICAL].isna().any(axis=1).sum()
    if n_missing > 0:
        print(f"WARNING: {n_missing}/{len(merged)} rows have missing metadata after join "
              f"-- check image_path vs image filename format matches exactly")
    assert len(merged) == len(split_df), "join changed row count -- duplicate or missing keys in img_info"
    return merged

# Metadata is already in the split CSVs — no separate img_info.xlsx join needed
train_meta = train_df.copy()
val_meta   = val_df.copy()
test_meta  = test_df.copy()

print(train_meta[META_NUMERIC + META_CATEGORICAL].isna().sum())

gestational_age    0
birth_weight       0
gender             0
plurality          0
delivery_mode      0
dtype: int64


In [5]:
print(train_df.columns.tolist())

['image', 'patient_id', 'gender', 'gestational_age', 'birth_weight', 'plurality', 'delivery_mode', 'categories']


## Build metadata feature vectors
Numeric fields -> z-score (fit on train only). Categorical fields -> one-hot (fit on train only, unseen categories at val/test map to all-zero).

In [6]:
def build_metadata_matrix(train_meta, val_meta, test_meta, numeric_cols, categorical_cols):
    """Fit scalers/encoders on train only, transform all three splits.
    Input:  train/val/test_meta DataFrames with numeric_cols + categorical_cols present
    Output: (M_train, M_val, M_test) float32 arrays, same row count as each input df
    """
    num_scaler = StandardScaler().fit(train_meta[numeric_cols].fillna(train_meta[numeric_cols].median()))
    cat_encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False).fit(
        train_meta[categorical_cols].astype(str))

    def transform(df):
        num_filled = df[numeric_cols].fillna(train_meta[numeric_cols].median())
        num_part = num_scaler.transform(num_filled)
        cat_part = cat_encoder.transform(df[categorical_cols].astype(str))
        return np.concatenate([num_part, cat_part], axis=1).astype(np.float32)

    return transform(train_meta), transform(val_meta), transform(test_meta)

M_train, M_val, M_test = build_metadata_matrix(train_meta, val_meta, test_meta, META_NUMERIC, META_CATEGORICAL)
print("metadata feature dim:", M_train.shape[1])
print("train/val/test metadata shapes:", M_train.shape, M_val.shape, M_test.shape)

metadata feature dim: 9
train/val/test metadata shapes: (1411, 9) (289, 9) (320, 9)


## Metadata MLP + fusion
Metadata MLP maps the small metadata vector into an embedding of the same size as the image embedding (512), so concatenation gives a 1024-d fused vector, then fusion layers compress it back down.
This module is frozen-image + trainable metadata/fusion only -- no image encoder weights are touched (per plan, EyeCLIP stays frozen until Phase 6/LoRA).

In [7]:
import torch
import torch.nn as nn

torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
IMG_DIM = X_train_img.shape[1]     # 512, from EyeCLIP
META_DIM = M_train.shape[1]
NUM_CLASSES = len(np.unique(y_train))

class MetadataMLP(nn.Module):
    """metadata vector [B, META_DIM] -> metadata embedding [B, IMG_DIM]"""
    def __init__(self, meta_dim, out_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(meta_dim, 64), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(64, out_dim),
        )
    def forward(self, m):
        return self.net(m)

class FusionClassifier(nn.Module):
    """(image_emb [B, IMG_DIM], metadata_emb [B, IMG_DIM]) -> class logits [B, NUM_CLASSES]
    Fusion: concat -> linear -> ReLU -> dropout -> linear (per plan step 16)
    """
    def __init__(self, img_dim, meta_dim, num_classes):
        super().__init__()
        self.meta_mlp = MetadataMLP(meta_dim, img_dim)
        self.fusion = nn.Sequential(
            nn.Linear(img_dim * 2, 256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, num_classes),
        )
    def forward(self, img_emb, meta_raw):
        meta_emb = self.meta_mlp(meta_raw)
        fused = torch.cat([img_emb, meta_emb], dim=1)
        return self.fusion(fused)

## Train fusion classifier (image+metadata -> 8-class, full train set)
Image embeddings are pre-extracted and frozen (no gradient) -- only MetadataMLP + fusion layers are trained. This mirrors Phase 1's linear-classifier-on-frozen-embeddings setup, but with metadata added.

In [8]:
def to_tensor(x, dtype=torch.float32):
    return torch.tensor(x, dtype=dtype, device=DEVICE)

Xi_train, Xi_val, Xi_test = to_tensor(X_train_img), to_tensor(X_val_img), to_tensor(X_test_img)
Xm_train, Xm_val, Xm_test = to_tensor(M_train), to_tensor(M_val), to_tensor(M_test)
Yt_train = to_tensor(y_train, dtype=torch.long)
Yt_val   = to_tensor(y_val,   dtype=torch.long)
Yt_test  = to_tensor(y_test,  dtype=torch.long)

model = FusionClassifier(IMG_DIM, META_DIM, NUM_CLASSES).to(DEVICE)

# class weights from train label counts -- imbalance is severe (Stage 5 n=5 vs normal n~300+)
class_counts = np.bincount(y_train, minlength=NUM_CLASSES)
class_weights = to_tensor(1.0 / np.maximum(class_counts, 1))
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

N_EPOCHS = 100
best_val_f1 = -1.0
best_state = None

for epoch in range(N_EPOCHS):
    model.train()
    optimizer.zero_grad()
    logits = model(Xi_train, Xm_train)
    loss = criterion(logits, Yt_train)
    loss.backward()
    optimizer.step()

    model.eval()
    with torch.no_grad():
        val_logits = model(Xi_val, Xm_val)
        val_preds = val_logits.argmax(dim=1).cpu().numpy()
        _, _, val_f1, _ = precision_recall_fscore_support(
            y_val, val_preds, average=None, zero_division=0)
        val_macro_f1 = val_f1.mean()

    if val_macro_f1 > best_val_f1:
        best_val_f1 = val_macro_f1
        best_state = {k: v.clone() for k, v in model.state_dict().items()}

    if epoch % 10 == 0:
        print(f"epoch {epoch:3d}  train_loss={loss.item():.4f}  val_macro_f1={val_macro_f1:.4f}")

model.load_state_dict(best_state)  # model selection on val, never on test
print(f"\nbest val macro-F1: {best_val_f1:.4f}")

epoch   0  train_loss=2.0876  val_macro_f1=0.0749
epoch  10  train_loss=1.9426  val_macro_f1=0.1633
epoch  20  train_loss=1.9036  val_macro_f1=0.1597
epoch  30  train_loss=1.8770  val_macro_f1=0.1714
epoch  40  train_loss=1.8366  val_macro_f1=0.1670
epoch  50  train_loss=1.8267  val_macro_f1=0.1828
epoch  60  train_loss=1.8127  val_macro_f1=0.1972
epoch  70  train_loss=1.7718  val_macro_f1=0.2005
epoch  80  train_loss=1.7793  val_macro_f1=0.1755
epoch  90  train_loss=1.7268  val_macro_f1=0.1818

best val macro-F1: 0.2083


## Evaluate on test set (image+metadata vs Phase 1 image-only baseline)

In [9]:
model.eval()
with torch.no_grad():
    test_logits = model(Xi_test, Xm_test)
    test_preds = test_logits.argmax(dim=1).cpu().numpy()

acc = accuracy_score(y_test, test_preds)
classes_sorted = sorted(np.unique(y_train))
precision, recall, f1, support = precision_recall_fscore_support(
    y_test, test_preds, average=None, labels=classes_sorted, zero_division=0)
macro_f1 = f1.mean()

print(f"Image+Metadata -- Test accuracy: {acc:.4f}   Macro-F1: {macro_f1:.4f}")
print(f"Phase 1 (image-only) for reference -- accuracy: 0.3063   Macro-F1: 0.2469\n")

print("Per-class:")
for c, p, r, f, s in zip(classes_sorted, precision, recall, f1, support):
    print(f"  class {c}  P={p:.3f}  R={r:.3f}  F1={f:.3f}  n={s}")

print("\nConfusion matrix:")
print(confusion_matrix(y_test, test_preds, labels=classes_sorted))

pd.DataFrame([{"representation": "image_only_phase1", "accuracy": 0.3063, "macro_f1": 0.2469},
              {"representation": "image_plus_metadata", "accuracy": acc, "macro_f1": macro_f1}]
            ).to_csv("phase3_baseline_comparison.csv", index=False)

Image+Metadata -- Test accuracy: 0.3187   Macro-F1: 0.2174
Phase 1 (image-only) for reference -- accuracy: 0.3063   Macro-F1: 0.2469

Per-class:
  class 0  P=0.152  R=0.200  F1=0.172  n=25
  class 1  P=0.263  R=0.185  F1=0.217  n=54
  class 2  P=0.283  R=0.351  F1=0.313  n=37
  class 3  P=0.290  R=0.173  F1=0.217  n=52
  class 4  P=0.167  R=0.184  F1=0.175  n=38
  class 5  P=0.000  R=0.000  F1=0.000  n=12
  class 6  P=0.000  R=0.000  F1=0.000  n=5
  class 7  P=0.699  R=0.598  F1=0.644  n=97

Confusion matrix:
[[ 5  5  1  1  5  1  2  5]
 [ 8 10  5  7 13  5  5  1]
 [ 1  4 13  6  2  0  3  8]
 [ 6  8 12  9  5  2  4  6]
 [ 5  4  8  5  7  3  2  4]
 [ 1  1  3  2  3  0  1  1]
 [ 2  0  0  0  3  0  0  0]
 [ 5  6  4  1  4 11  8 58]]


## Few-shot with fused representations (plan step 17)
Same nearest-prototype method as Phase 2, but prototypes are built on **fused** embeddings (image_emb + metadata_emb concatenated, taken from the trained fusion model's intermediate representation) instead of raw image embeddings.
Reuses the exact same `build_prototypes` / `nearest_prototype_predict` logic and seeds as Phase 2 for a fair comparison.

In [10]:
def build_prototypes(X, y, classes, k, rng):
    """Sample k examples per class and average -- identical to Phase 2's function."""
    prototypes = []
    for c in classes:
        idx = np.where(y == c)[0]
        n_take = min(k, len(idx))
        chosen = rng.choice(idx, size=n_take, replace=False)
        prototypes.append(X[chosen].mean(axis=0))
    return np.stack(prototypes)

def nearest_prototype_predict(X, prototypes, classes, metric="cosine"):
    """Identical to Phase 2's function."""
    if metric == "cosine":
        X_n = X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-8)
        P_n = prototypes / (np.linalg.norm(prototypes, axis=1, keepdims=True) + 1e-8)
        nearest_idx = (X_n @ P_n.T).argmax(axis=1)
    else:
        dists = np.linalg.norm(X[:, None, :] - prototypes[None, :, :], axis=2)
        nearest_idx = dists.argmin(axis=1)
    return np.array([classes[i] for i in nearest_idx])

# extract the model's fused representation (post-concat, pre-final-classification-layer)
# by running the meta_mlp + concat, but stopping before the last fusion layer
@torch.no_grad()
def get_fused_repr(img_t, meta_t):
    meta_emb = model.meta_mlp(meta_t)
    fused = torch.cat([img_t, meta_emb], dim=1)
    return model.fusion[0](fused).cpu().numpy()  # after first Linear(1024,256), before ReLU/dropout/final

model.eval()
F_train = get_fused_repr(Xi_train, Xm_train)
F_test  = get_fused_repr(Xi_test,  Xm_test)
F_val = get_fused_repr(Xi_val, Xm_val)
SHOTS = [1, 3, 5]
SEEDS = [0, 1, 2, 3, 4]
classes_sorted = sorted(np.unique(y_train))

fusion_fewshot_results = []
for k in SHOTS:
    for seed in SEEDS:
        rng = np.random.default_rng(seed)
        prototypes = build_prototypes(F_train, y_train, classes_sorted, k, rng)
        preds = nearest_prototype_predict(F_test, prototypes, classes_sorted, metric="cosine")
        acc_k = accuracy_score(y_test, preds)
        _, _, f1_k, _ = precision_recall_fscore_support(
            y_test, preds, labels=classes_sorted, average=None, zero_division=0)
        fusion_fewshot_results.append({"shot": k, "seed": seed, "accuracy": acc_k, "macro_f1": f1_k.mean()})

fusion_fewshot_df = pd.DataFrame(fusion_fewshot_results)
fusion_summary = fusion_fewshot_df.groupby("shot").agg(
    accuracy_mean=("accuracy", "mean"), accuracy_std=("accuracy", "std"),
    macro_f1_mean=("macro_f1", "mean"), macro_f1_std=("macro_f1", "std"),
).reset_index()
print(fusion_summary.to_string(index=False))
fusion_summary.to_csv("phase3_fewshot_fused_results.csv", index=False)
np.savez("fused_embeddings.npz", F_train=F_train, F_val=F_val, F_test=F_test)

 shot  accuracy_mean  accuracy_std  macro_f1_mean  macro_f1_std
    1       0.178125      0.082827       0.134565      0.041467
    3       0.266250      0.035314       0.180571      0.013226
    5       0.229375      0.081057       0.147911      0.034796


## Final comparison table (plan step 18: image-only vs image+metadata across shots)

In [11]:
# fill in your actual Phase 2 image-only few-shot numbers here for the side-by-side table
phase2_image_only = pd.DataFrame([
    {"shot": 1, "representation": "image_only", "accuracy_mean": 0.154375, "macro_f1_mean": 0.127261},
    {"shot": 3, "representation": "image_only", "accuracy_mean": 0.139375, "macro_f1_mean": 0.109069},
    {"shot": 5, "representation": "image_only", "accuracy_mean": 0.133125, "macro_f1_mean": 0.097823},
])

phase3_fused = fusion_summary[["shot", "accuracy_mean", "macro_f1_mean"]].copy()
phase3_fused["representation"] = "image_plus_metadata"

comparison = pd.concat([phase2_image_only, phase3_fused], ignore_index=True)
comparison = comparison.sort_values(["shot", "representation"])
print(comparison.to_string(index=False))
comparison.to_csv("phase3_image_vs_fused_fewshot.csv", index=False)

 shot      representation  accuracy_mean  macro_f1_mean
    1          image_only       0.154375       0.127261
    1 image_plus_metadata       0.178125       0.134565
    3          image_only       0.139375       0.109069
    3 image_plus_metadata       0.266250       0.180571
    5          image_only       0.133125       0.097823
    5 image_plus_metadata       0.229375       0.147911


## Done — Phase 3 deliverable
Record: (1) full-train image+metadata vs Phase 1 image-only baseline, (2) few-shot image+metadata vs Phase 2 image-only few-shot.
Do **not** proceed to Phase 4 (open-set) until both comparisons are recorded.